# 🗺️ Notebook 03 — Causal Risk Map & Thảo luận

**Khóa luận:** Phân tích Lan tỏa Rủi ro sử dụng NOTEARS Causal Discovery  
**Chương tương ứng:** Chương 4.6 — Causal Risk Map & Thảo luận kinh tế

---

## Mục tiêu
1. Vẽ Causal Risk Map tổng hợp 3 phương pháp
2. Phân tích hub/sink node — ai phát rủi ro, ai nhận
3. Subperiod analysis — cấu trúc DAG thay đổi qua COVID / crypto crash
4. So sánh với literature (kết quả có tương đồng không?)
5. Trả lời RQ1, RQ2, RQ3

## 0. Setup & Load kết quả từ Notebook 02

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.logger  import setup_logger
from src.utils.io_utils import ResultIO
from config import NODE_NAMES, MARKET_LABELS
setup_logger('KLTN')

plt.rcParams.update({'figure.dpi': 120})

# Load snapshot từ Notebook 02
io   = ResultIO('../reports')
data = io.load_snapshot()

results = data['results']    # {'Granger': ..., 'PC Algorithm': ..., 'NOTEARS': ...}
vol_df  = data['vol_df']

granger_result = results['Granger']
pc_result      = results['PC Algorithm']
notears_result = results['NOTEARS']

print('✅ Load snapshot OK')
for name, r in results.items():
    print(f'  {name:<15s}: {r.n_edges} edges | hub={r.hub_nodes(1)}')

---
## 1. Causal Risk Map tổng hợp 3 phương pháp

In [ ]:
from src.visualization.risk_map import plot_risk_map

fig = plot_risk_map(
    results    = results,
    node_names = NODE_NAMES,
    output_path= '../reports/figures/03_causal_risk_map.png',
    figsize    = (18, 6),
)
plt.show()
print('\n📌 Figure chính — Chương 4.6')

---
## 2. Phân tích Hub / Sink Node

In [ ]:
from src.visualization.risk_map import plot_degree_analysis

print('=' * 55)
print('  PHÂN TÍCH HUB / SINK NODE — NOTEARS ★')
print('=' * 55)

degree_df = notears_result.degree_table()
print(degree_df.to_string())

print('\n📌 Nhận xét:')
for market, row in degree_df.iterrows():
    label = MARKET_LABELS.get(market, market)
    role  = row['Role']
    out   = row['Out-degree']
    ins   = row['In-degree']
    net   = row['Net']
    if role == 'Source':
        print(f'  ▶ {label:<15s}: NGUỒN phát rủi ro (out={out}, net={net:+d})')
    elif role == 'Sink':
        print(f'  ◀ {label:<15s}: NHẬN rủi ro (in={ins}, net={net:+d})')
    else:
        print(f'  ◆ {label:<15s}: Trung lập (out={out}, in={ins})')

In [ ]:
fig = plot_degree_analysis(
    result      = notears_result,
    node_names  = NODE_NAMES,
    title       = 'NOTEARS ★ — Degree Analysis: Risk Emitter vs Receiver',
    output_path = '../reports/figures/03_notears_degree_analysis.png',
)
plt.show()

---
## 3. Subperiod Analysis — COVID vs Crypto Crash

In [ ]:
from src.pipeline.subperiod_pipeline import SubperiodPipeline

subpipe = SubperiodPipeline(
    vol_df     = vol_df,
    node_names = NODE_NAMES,
    threshold  = 0.2,
    lambda1    = 0.1,
)
period_df = subpipe.run()
subpipe.print_summary()

In [ ]:
# Bảng tóm tắt đẹp
print('\n📋 Kết quả theo giai đoạn:')
period_df.style.background_gradient(cmap='YlOrRd', subset=['Số cạnh', 'Graph Density'])

In [ ]:
# Edge stability heatmap
from src.visualization.risk_map import plot_edge_stability

freq_df = subpipe.edge_frequency_matrix()
fig = plot_edge_stability(
    freq_matrix = freq_df.values,
    node_names  = NODE_NAMES,
    output_path = '../reports/figures/03_edge_stability.png',
)
plt.show()

print('\n📌 Cạnh ổn định (tần suất ≥ 50%):')
stable = subpipe.stable_edges(min_freq=0.5)
for e in stable:
    src = MARKET_LABELS.get(e['source'], e['source'])
    tgt = MARKET_LABELS.get(e['target'], e['target'])
    print(f'  {src:<15s} → {tgt:<15s}: freq={e["frequency"]} | {e["stability"]}')

In [ ]:
# Crisis vs Normal SHD
shd_df = subpipe.crisis_comparison()
print('\n📋 SHD giữa các giai đoạn:')
shd_df.style.background_gradient(cmap='Reds')

In [ ]:
# Hub evolution
hub_df = subpipe.hub_evolution()
print('\n📋 Out-degree theo giai đoạn (hub node evolution):')

fig, ax = plt.subplots(figsize=(11, 5))
from config import MARKET_COLORS, MARKET_LABELS as ML
for market in NODE_NAMES:
    if market in hub_df.columns:
        ax.plot(
            hub_df.index, hub_df[market],
            marker='o', lw=2, ms=7,
            color=MARKET_COLORS.get(market, '#aaa'),
            label=ML.get(market, market),
        )

ax.set_xlabel('Giai đoạn', fontsize=11)
ax.set_ylabel('Out-degree', fontsize=11)
ax.set_title('Hub Node Evolution — Out-degree qua các Giai đoạn', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('../reports/figures/03_hub_evolution.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Thảo luận kinh tế — Trả lời RQ1, RQ2, RQ3

In [ ]:
print('=' * 65)
print('  TRẢ LỜI CÂU HỎI NGHIÊN CỨU')
print('=' * 65)

# RQ1: Cấu trúc DAG nhân quả
print('\n🔵 RQ1: Cấu trúc DAG nhân quả tồn tại giữa 5 thị trường?')
edges = notears_result.edge_list()
print(f'  NOTEARS học được {len(edges)} quan hệ nhân quả có hướng:')
for e in edges:
    src = MARKET_LABELS.get(e['source'], e['source'])
    tgt = MARKET_LABELS.get(e['target'], e['target'])
    print(f'    {src:<15s} → {tgt:<15s} (w={e["weight"]:+.3f})')

# RQ2: So sánh hiệu quả
print('\n🔵 RQ2: NOTEARS có hiệu quả hơn PC và Granger?')
from src.evaluation.comparator import MethodComparator
cmp = MethodComparator(granger_result.binary_matrix, n_bootstrap=500)
cmp.add('PC Algorithm', pc_result)
cmp.add('NOTEARS ★',   notears_result)
best = cmp.best_method('F1')
print(f'  Phương pháp tốt nhất (F1): {best}')
print(cmp.raw_scores().to_string())

# RQ3: Hub node
print('\n🔵 RQ3: Thị trường nào là nguồn phát rủi ro chính?')
hubs  = notears_result.hub_nodes(top_k=2)
sinks = notears_result.sink_nodes(top_k=2)
for h, d in hubs:
    print(f'  ▶ {MARKET_LABELS.get(h,h):<15s}: out-degree={d} (PHÁT rủi ro)')
for s, d in sinks:
    print(f'  ◀ {MARKET_LABELS.get(s,s):<15s}: in-degree={d} (NHẬN rủi ro)')

---
## 5. Export Full Report

In [ ]:
from src.utils.io_utils import ResultIO
from config import NOTEARS_LAMBDA1, NOTEARS_THRESHOLD, GRANGER_MAX_LAG, PC_ALPHA

io      = ResultIO('../reports')
cmp_df  = cmp.comparison_table()

# Excel báo cáo tổng hợp
report_path = io.export_full_report(
    results      = results,
    comparison_df= cmp_df,
    config_dict  = {
        'NOTEARS lambda1'  : NOTEARS_LAMBDA1,
        'NOTEARS threshold': NOTEARS_THRESHOLD,
        'Granger max_lag'  : GRANGER_MAX_LAG,
        'PC alpha'         : PC_ALPHA,
        'Bootstrap n'      : 1000,
        'Data period'      : '2018-01-01 ~ 2024-12-31',
        'Rolling window'   : '21 ngày',
    }
)
print(f'\n✅ Full report: {report_path}')

# Lưu danh sách ảnh
import os
figs = sorted(os.listdir('../reports/figures'))
print(f'\n📂 Figures ({len(figs)} files):')
for f in figs:
    print(f'  reports/figures/{f}')